In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Chargement avec chemins relatifs
df_rte = pd.read_csv("../data/df_rte.csv")
df_meteo = pd.read_csv("../data/df_meteo.csv")

print("=== RTE ===")
print(df_rte.info())
print(df_rte.head(2))

print("\n=== METEO ===")
print(df_meteo.info())
print(df_meteo.head(2))

=== RTE ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210408 entries, 0 to 210407
Data columns (total 9 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   date_heure_utc        210408 non-null  object 
 1   consommation          105203 non-null  float64
 2   solaire               105203 non-null  float64
 3   eolien                105203 non-null  float64
 4   nucleaire             105203 non-null  float64
 5   hydraulique           105203 non-null  float64
 6   gaz                   105203 non-null  float64
 7   ech_physiques         105203 non-null  float64
 8   charge_residuelle_mw  105203 non-null  float64
dtypes: float64(8), object(1)
memory usage: 14.4+ MB
None
              date_heure_utc  consommation  solaire  eolien  nucleaire  \
0  2020-01-01 00:00:00+00:00       63943.0     -1.0  3203.0    48970.0   
1  2020-01-01 00:15:00+00:00           NaN      NaN     NaN        NaN   

   hydraulique     gaz  e

In [6]:
# 1. Voir quelles sont les villes présentes dans df_meteo
print("Villes dans le dataset météo :", df_meteo["ville"].unique())

# 2. Nettoyage de df_rte : on enlève les lignes vides et on garde les heures pleines
df_rte_clean = df_rte.dropna(subset=["consommation"]).copy()
df_rte_clean["date_heure_utc"] = pd.to_datetime(df_rte_clean["date_heure_utc"])
df_rte_horaire = df_rte_clean[df_rte_clean["date_heure_utc"].dt.minute == 0].copy()

# 3. Agrégation de df_meteo : moyenne nationale par heure (des 3 villes)
df_meteo["date_heure_utc"] = pd.to_datetime(df_meteo["date_heure_utc"])
df_meteo_nat = (
    df_meteo.groupby("date_heure_utc", as_index=False)[
        ["wind_speed_100m", "shortwave_radiation", "temperature_2m"]
    ]
    .mean()
)

# 4. Jointure (merge) des deux tables sur la colonne date_heure_utc
df = pd.merge(df_rte_horaire, df_meteo_nat, on="date_heure_utc", how="inner")

# Vérification du résultat
print(f"Nombre d'heures après fusion : {len(df)}")
print(f"Période couverte : du {df['date_heure_utc'].min()} au {df['date_heure_utc'].max()}")
display(df.head(3))

Villes dans le dataset météo : ['Brest' 'Marseille' 'Paris']
Nombre d'heures après fusion : 52601
Période couverte : du 2020-01-01 00:00:00+00:00 au 2025-12-31 22:00:00+00:00


,date_heure_utc,consommation,solaire,eolien,nucleaire,hydraulique,gaz,ech_physiques,charge_residuelle_mw,wind_speed_100m,shortwave_radiation,temperature_2m
0,2020-01-01 00:00:00+00:00,63943.0,-1.0,3203.0,48970.0,9133.0,6257.0,-4889.0,60741.0,1.380000,0.0,4.900000
1,2020-01-01 01:00:00+00:00,63408.0,-1.0,2799.0,49764.0,8699.0,5337.0,-4464.0,60610.0,1.523333,0.0,4.833333
2,2020-01-01 02:00:00+00:00,60825.0,-1.0,2810.0,49527.0,7658.0,3660.0,-3457.0,58016.0,2.000000,0.0,4.600000
